# Complete all five detectors for the recorded FD001 Raw and AE-21 representations.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path

REVISION_ROOT = Path("revision_outputs")
DATA_DIR = Path("data")
PYTHON_EXE = str(RELEASE_CLASSICAL_PYTHON)


In [ ]:
import subprocess,sys,os,time,json,platform
from pathlib import Path
from datetime import datetime,timezone


def python_candidates(explicit=None):
    if explicit:
        return [str(Path(explicit).expanduser().absolute())]
    candidates=[Path(sys.executable)]
    bases={Path(sys.prefix),Path(sys.base_prefix),Path(sys.executable).parent.parent,
           Path('/opt/anaconda3'),Path('/opt/miniconda3'),Path.home()/'miniconda3',Path.home()/'anaconda3'}
    for base in sorted(bases):
        candidates.extend(sorted((base/'envs').glob('*/bin/python')))
    candidates.extend(sorted((Path.home()/'.conda/envs').glob('*/bin/python')))
    registry=Path.home()/'.conda/environments.txt'
    if registry.is_file():
        candidates.extend(Path(line.strip())/'bin/python' for line in registry.read_text().splitlines() if line.strip())
    for base in [Path.cwd(),Path.cwd().parent]:
        candidates.extend([base/'.venv/bin/python',base/'venv/bin/python'])

    unique=[]
    for path in candidates:
        value=str(path.expanduser().absolute())
        if path.is_file() and value not in unique:unique.append(value)
    return unique


def compatible_python(expected,out,explicit=None):
    code="""import json,platform,importlib.metadata as m
v={'python':platform.python_version()}
for key,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('tensorflow','tensorflow')]:
 try:v[key]=m.version(package)
 except m.PackageNotFoundError:v[key]='NOT_INSTALLED'
print(json.dumps(v))
"""
    reports=[]
    for executable in python_candidates(explicit):
        item={'executable':executable}
        try:
            proc=subprocess.run([executable,'-c',code],capture_output=True,text=True,timeout=20)
            item['metadata_returncode']=proc.returncode
            if proc.returncode!=0:
                item['status']='METADATA_FAILED';item['stderr_tail']=proc.stderr[-1500:];reports.append(item);continue
            versions=json.loads(proc.stdout.strip().splitlines()[-1]);item['versions']=versions
            if any(versions.get(k)!=v for k,v in expected.items()):
                item['status']='VERSIONS_DIFFER';reports.append(item);continue
            print('Eğitim sürümleriyle eşleşen Python kontrol ediliyor:',executable,flush=True)
            probe="import numpy, pandas; import tensorflow as tf; import sklearn; print('TF_PROBE_OK',float(tf.reduce_sum(tf.constant([1.,2.])).numpy()),flush=True)"
            proc=subprocess.run([executable,'-X','faulthandler','-c',probe],capture_output=True,text=True,timeout=60)
            item.update(probe_returncode=proc.returncode,probe_stdout_tail=proc.stdout[-1500:],probe_stderr_tail=proc.stderr[-3000:])
            if proc.returncode==0 and 'TF_PROBE_OK 3.0' in proc.stdout:
                item['status']='READY';reports.append(item);return executable,reports
            item['status']='IMPORT_OR_TENSOR_TEST_FAILED'
        except subprocess.TimeoutExpired:
            item['status']='PROBE_TIMEOUT'
        except (OSError,ValueError) as e:
            item.update(status='PROBE_ERROR',error=str(e))
        reports.append(item)
    return None,reports


import hashlib,statistics,csv,math,fcntl
from collections import defaultdict


def canonical_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode()).hexdigest()


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def read_json(path):return json.loads(Path(path).read_text())


def write_json(path,value):
    path=Path(path);temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False));temp.replace(path)


def require(ok,message):
    if not bool(ok):raise ValueError(message)


def safe_path(root,relative):
    path=(Path(root)/relative).resolve()
    require(path.is_relative_to(Path(root).resolve()),'Path escapes permitted directory: '+str(relative))
    return path


def verify_manifest(folder,manifest):
    require(bool(manifest),'Empty output hash manifest')
    for name,h in manifest.items():
        path=safe_path(folder,name)
        require(path.is_file() and digest(path)==h,'Output missing/changed: '+str(path))


def verify_source(root,data,job):
    for rel,h in job['known_hashes'].items():
        p=safe_path(root,rel);require(p.is_file() and digest(p)==h,'Source missing/changed: '+str(p))
    require(digest(Path(data)/'train_FD001.txt')==job['train_sha256'],'FD001 source hash changed')


def validate_result(result,job,addendum_hash,full_hash=None):
    require(result['status'] in ['PASS_FOLD','PASS_PILOT'],'Fold did not pass: '+result.get('status','missing'))
    require((result['seed'],result['fold'])==(job['seed'],job['fold']),'Fold identity mismatch')
    require(result['addendum_sha256']==addendum_hash,'Detector protocol mismatch')
    require(result['versions']==job['expected_versions'],'Fold runtime versions differ')
    if full_hash is not None:require(result.get('full_plan_sha256')==full_hash,'Full-run plan mismatch')
    require(result['baseline_reproduction_status']=='PASS_SINGLE_FOLD_REPRODUCTION','Baseline not reproduced')
    require(result['AE_training_performed'] is False and result['test_files_read'] is False,'Unexpected training/test scope')
    require(set(result['detectors'])=={'DBSCAN','IF','LOF','OC-SVM','kNN'},'Detector set incomplete')
    trials=result['selection_trials']
    require(len(trials)==36 and all(t['status']=='VALID' for t in trials),'Incomplete candidate search')
    for rep in ['Raw','AE_latent21']:
        for method in ['DBSCAN','OC-SVM']:
            group=[t for t in trials if t['representation']==rep and t['method']==method]
            require(len(group)==9 and sorted(t['candidate_order'] for t in group)==list(range(9)),'Candidate coverage mismatch')
            best=max(group,key=lambda t:t['selection_AP']);chosen=result['selected_configurations'][rep+'/'+method]
            require(chosen['params']==best['params'] and chosen['candidate_order']==best['candidate_order'] and chosen['selection_AP']==best['selection_AP'],'Selection not supported by trial log')
    rows=result['primary_results']+result['sensitivity_results']
    expected={(rep,m,w) for rep in ['Raw','AE_latent21'] for m in ['DBSCAN','IF','LOF','OC-SVM','kNN'] for w in [20,30,40]}
    expected|={('AE_reconstruction','MSE',w) for w in [20,30,40]}
    keys=[(r['representation'],r['method'],r['RUL_window']) for r in rows]
    require(len(keys)==33 and len(set(keys))==33 and set(keys)==expected,'Expected all 33 metric rows')
    for old in job['approved_primary_results']:
        now=next(r for r in result['primary_results'] if (r['representation'],r['method'])==(old['representation'],old['method']))
        require(all(now[k]==v for k,v in old.items()),'Historical primary result changed')
    counts=defaultdict(set);decisions=defaultdict(set)
    for r in rows:
        require((r['seed'],r['fold'])==(job['seed'],job['fold']),'Row fold identity mismatch')
        require(all(math.isfinite(r[k]) and 0<=r[k]<=1 for k in METRICS),'Invalid metric')
        tp,fp,tn,fn=[r[k] for k in ['TP','FP','TN','FN']]
        require(all(isinstance(x,int) and x>=0 for x in [tp,fp,tn,fn]),'Invalid confusion counts')
        counts[r['RUL_window']].add((tp+fn,tn+fp))
        p=tp/(tp+fp) if tp+fp else 0;rec=tp/(tp+fn) if tp+fn else 0
        f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
        f0=2*tn/(2*tn+fp+fn) if 2*tn+fp+fn else 0
        for key,value in [('precision',p),('recall',rec),('F1',f1),('Macro_F1',(f1+f0)/2)]:
            require(math.isclose(r[key],value,abs_tol=1e-12,rel_tol=0),'Confusion-derived metric mismatch')
        require(math.isfinite(r['threshold']),'Nonfinite threshold')
        decisions[(r['representation'],r['method'])].add((r['threshold'],canonical_hash(r['selected_params'])))
    require(all(len(x)==1 for x in counts.values()),'Reference class counts differ across methods')
    require(all(len(x)==1 for x in decisions.values()),'Sensitivity reused different thresholds/parameters')
    return rows


METRICS=['AP','ROC_AUC','precision','recall','F1','Macro_F1']


def aggregate_results(results,jobs):
    require(len(results)==len(jobs)==30,'Thirty completed fold/seed blocks required')
    rows=[r for result in results for r in result['primary_results']+result['sensitivity_results']]
    expected_pairs={(seed,fold) for seed in [42,43,44] for fold in range(1,11)}
    keys=[(r['representation'],r['method'],r['RUL_window'],r['seed'],r['fold']) for r in rows]
    require(len(rows)==990 and len(set(keys))==990,'Expected 990 unique metric rows')
    groups=defaultdict(list)
    for r in rows:groups[(r['representation'],r['method'],r['RUL_window'])].append(r)
    require(len(groups)==33,'Expected 33 representation/method/window groups')
    overall=[];per_seed=[]
    for (rep,method,window),group in sorted(groups.items()):
        require(len(group)==30 and {(r['seed'],r['fold']) for r in group}==expected_pairs,'Missing fold/seed')
        label='DBSCAN (core-distance)' if method=='DBSCAN' else method
        item={'representation':rep,'method':method,'method_display_name':label,'RUL_window':window,'runs':30}
        for metric in METRICS:
            values=[r[metric] for r in group];item[metric+'_mean']=statistics.mean(values);item[metric+'_descriptive_sd']=statistics.stdev(values)
        overall.append(item)
        for seed in [42,43,44]:
            subset=[r for r in group if r['seed']==seed]
            each={'representation':rep,'method':method,'RUL_window':window,'seed':seed}
            for metric in METRICS:
                values=[r[metric] for r in subset];each[metric+'_fold_mean']=statistics.mean(values);each[metric+'_fold_sd']=statistics.stdev(values)
            per_seed.append(each)
    primary=[r for r in rows if r['RUL_window']==30];index={(r['representation'],r['method'],r['seed'],r['fold']):r for r in primary}
    paired=[]
    for method in ['DBSCAN','IF','LOF','OC-SVM','kNN']:
        item={'method':method,'left':'AE_latent21','right':'Raw','pairs':30,'direction':'AE minus Raw'}
        for metric in ['AP','F1','Macro_F1']:
            delta=[index[('AE_latent21',method,s,f)][metric]-index[('Raw',method,s,f)][metric] for s,f in sorted(expected_pairs)]
            item['delta_'+metric+'_mean']=statistics.mean(delta);item['delta_'+metric+'_descriptive_sd']=statistics.stdev(delta)
            item['positive_'+metric+'_runs']=sum(v>1e-12 for v in delta);item['negative_'+metric+'_runs']=sum(v< -1e-12 for v in delta);item['tied_'+metric+'_runs']=sum(abs(v)<=1e-12 for v in delta)
        paired.append(item)
    return {'primary_results':[r for r in overall if r['RUL_window']==30],
            'sensitivity_results':[r for r in overall if r['RUL_window']!=30],
            'per_seed_results':per_seed,'paired_raw_vs_latent':paired,
            'fold_primary_results':primary,'fold_sensitivity_results':[r for r in rows if r['RUL_window']!=30]}


def write_csv(path,rows):
    fields=list(dict.fromkeys(k for row in rows for k in row))
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        writer.writerows({k:json.dumps(v,ensure_ascii=False,sort_keys=True) if isinstance(v,(dict,list)) else v for k,v in r.items()} for r in rows)


def find_approved_pilot(root,approved_hash):
    candidates=sorted((root/'03B_five_method_pilot').glob('*/five_method_pilot_summary.json'))
    for path in candidates:
        try:result=read_json(path)
        except (OSError,ValueError):continue
        if canonical_hash(result)==approved_hash:
            verify_manifest(path.parent,result['output_sha256']);return path,result
    raise FileNotFoundError('Paylaştığınız 03B pilot çıktısı bulunamadı. Önceki 03B çıktı klasörünü koruyun; bu raporu paylaşın.')


def run_child(executable,worker,configpath,attempt,timeout):
    logpath=attempt/'worker.log';start=time.monotonic();last=start;position=0;timed_out=False
    with logpath.open('w') as log:
        process=subprocess.Popen([executable,'-u','-X','faulthandler',str(worker),str(configpath)],stdout=log,stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                time.sleep(1)
                if time.monotonic()-start>timeout:
                    process.kill();process.wait();timed_out=True;break
                with logpath.open(errors='replace') as reader:
                    reader.seek(position);new=reader.read();position=reader.tell()
                if new:print(new,end='',flush=True)
                if time.monotonic()-last>=30:
                    print('Bu koşuda geçen süre:',round(time.monotonic()-start),'sn',flush=True);last=time.monotonic()
        except KeyboardInterrupt:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
    with logpath.open(errors='replace') as reader:reader.seek(position);new=reader.read()
    if new:print(new,end='',flush=True)
    marker=attempt/'fold_completion_summary.json'
    if timed_out or process.returncode!=0 or not marker.is_file():
        return {'status':'WORKER_TIMEOUT' if timed_out else 'WORKER_FAILED','worker_returncode':process.returncode,'log_tail':logpath.read_text(errors='replace')[-10000:]},marker
    return read_json(marker),marker


def run_full(root,data,template,worker_source,explicit_python=None):
    root=Path(root).expanduser().resolve();data=Path(data).expanduser().resolve()
    require(root.is_dir(),'REVISION_ROOT bulunamadı: '+str(root))
    design={**template,'worker_source_sha256':hashlib.sha256(worker_source.encode()).hexdigest()}
    plan_hash=canonical_hash(design);run=root/'05B_FD001_five_method_completion'/plan_hash[:20];run.mkdir(parents=True,exist_ok=True)
    summary_path=run/'cv_completion_summary.json';jobs=template['jobs'];results=[];sources=[];executed=0
    started=time.monotonic();active=None
    lock=(run/'run.lock').open('a')
    try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:lock.close();raise RuntimeError('Bu plan başka bir süreçte çalışıyor; ikinci bir Run All başlatmayın.')
    report={'stage':'05B_FD001_five_method_completion','status':'RUNNING','full_plan_sha256':plan_hash,
            'addendum_sha256':template['addendum_sha256'],'AE_training_performed':False,'test_files_read':False,
            'previous_experiment_outputs_modified':False,'expected_jobs':30,'completed_jobs':0,
            'folds':list(range(1,11)),'seeds':[42,43,44],'representations':['Raw','AE_latent21'],
            'detectors':['DBSCAN','IF','LOF','OC-SVM','kNN'],'DBSCAN_variant':'DBSCAN_core_distance_v1',
            'source_pilot_canonical_sha256':template['pilot_canonical_sha256'],
            'versions_expected':template['expected_versions']}
    try:
        planpath=run/'full_run_plan.json'
        if planpath.exists():require(read_json(planpath)==design,'Saved plan differs')
        else:write_json(planpath,design)
        worker=run/'fold_worker.py'
        if worker.exists():require(worker.read_text()==worker_source,'Saved worker differs')
        else:worker.write_text(worker_source)
        pilotpath,pilot=find_approved_pilot(root,template['pilot_canonical_sha256'])
        validate_result(pilot,jobs[0],template['addendum_sha256'])
        executable,probes=compatible_python(template['expected_versions'],run,explicit_python)
        report['environment_checks']=probes;report['worker_executable']=executable
        require(executable is not None,'Çalışan ve sürümleri eşleşen Python bulunamadı.')
        for i,job in enumerate(jobs,1):
            active={'seed':job['seed'],'fold':job['fold']}
            verify_source(root,data,job)
            folder=run/f"seed_{job['seed']}"/f"fold_{job['fold']:02d}";folder.mkdir(parents=True,exist_ok=True)
            done=folder/'done.json'
            print(f"\n[{i}/30] seed={job['seed']} fold={job['fold']}",flush=True)
            if (job['seed'],job['fold'])==(42,1):
                result=pilot;sourcepath=pilotpath;kind='approved_03B_pilot_reused'
                print('Doğrulanmış 03B pilotu yeniden kullanılacak.',flush=True)
            elif done.is_file():
                pointer=read_json(done);require(pointer['full_plan_sha256']==plan_hash,'Cache plan differs')
                sourcepath=safe_path(folder,pointer['summary_relative_path'])
                require(digest(sourcepath)==pointer['summary_sha256'],'Cached summary changed')
                result=read_json(sourcepath);kind='completed_05B_job_reused'
                verify_manifest(sourcepath.parent,result['output_sha256'])
                validate_result(result,job,template['addendum_sha256'],plan_hash)
                print('Tamamlanmış koşu doğrulandı; yeniden eğitim yapılmıyor.',flush=True)
            else:
                attempt=folder/datetime.now(timezone.utc).strftime('attempt_%Y%m%dT%H%M%S_%fZ');attempt.mkdir(exist_ok=False)
                config={**job,'revision_root':str(root),'data_directory':str(data),'output_directory':str(attempt),
                        'expected_versions':template['expected_versions'],'five_method_addendum':template['five_method_addendum'],
                        'addendum_sha256':template['addendum_sha256'],'full_plan_sha256':plan_hash}
                cfg=attempt/'fold_config.json';write_json(cfg,config)
                executed+=1
                result,sourcepath=run_child(executable,worker,cfg,attempt,template['five_method_addendum']['resources']['worker_timeout_seconds'])
                report['last_job_status']=result.get('status');report['last_job_summary']=str(sourcepath)
                require(result.get('status')=='PASS_FOLD','Koşu tamamlanamadı: '+str(result.get('status'))+'; '+str(result.get('error',result.get('log_tail',''))))
                verify_manifest(attempt,result['output_sha256']);validate_result(result,job,template['addendum_sha256'],plan_hash)
                write_json(done,{'full_plan_sha256':plan_hash,'summary_relative_path':str(sourcepath.relative_to(folder)),
                                 'summary_sha256':digest(sourcepath)})
                kind='new_05B_job_completed'
            validate_result(result,job,template['addendum_sha256'],None if kind=='approved_03B_pilot_reused' else plan_hash)
            results.append(result);sources.append({**active,'kind':kind,'summary_path':str(sourcepath),'summary_sha256':digest(sourcepath)})
            report.update(completed_jobs=len(results),job_sources=sources,new_jobs_started_this_invocation=executed)
            write_json(summary_path,report)
        aggregates=aggregate_results(results,jobs)

        for old in template['baseline_aggregate_primary']:
            now=next(x for x in aggregates['primary_results'] if (x['representation'],x['method'])==(old['representation'],old['method']))
            for metric in METRICS:
                for suffix in ['_mean','_descriptive_sd']:
                    require(math.isclose(now[metric+suffix],old[metric+suffix],rel_tol=0,abs_tol=1e-12),'Historical aggregate changed')
        all_rows=aggregates['fold_primary_results']+aggregates['fold_sensitivity_results']
        write_csv(run/'all_fold_metrics.csv',all_rows)
        write_csv(run/'aggregate_metrics.csv',aggregates['primary_results']+aggregates['sensitivity_results'])
        write_csv(run/'per_seed_metrics.csv',aggregates['per_seed_results'])
        write_csv(run/'paired_differences.csv',aggregates['paired_raw_vs_latent'])
        fixed=[{**r,'seed':res['seed'],'fold':res['fold']} for res in results for r in res['fixed_radius_diagnostics']]
        write_csv(run/'dbscan_fixed_radius_diagnostics.csv',fixed)
        selected=[{'seed':r['seed'],'fold':r['fold'],'configurations':r['selected_configurations']} for r in results]
        report.update(aggregates,status='PASS_FD001_BASELINE_5_METHODS',completed_jobs=30,primary_groups=11,
                      metric_rows=990,new_method_primary_rows=120,selected_configurations=selected,
                      fold_selection_trials=[{**t,'seed':res['seed'],'fold':res['fold']} for res in results for t in res['selection_trials']],
                      versions=results[0]['versions'],
                      fixed_radius_diagnostics=fixed,pilot_reused=True,new_AE_training_runs=0,
                      source_baseline_primary_preserved=True,source_baseline_aggregate_preserved=True,
                      statistical_tests_performed=False,
                      aggregation='equal-weight fold/seed means and sample SD; no pooling of cross-fold scores',
                      limits=['Only FD001 Raw/AE21 baseline is completed, not AE/PCA ablations or FD002/FD003.',
                              'Fold/seed repetitions share data; SD is descriptive, not a confidence interval.',
                              'Raw DBSCAN/OC-SVM/LOF/kNN repeated seeds are not independent fits to new data.',
                              'DBSCAN is the disclosed fixed-core distance variant; candidate grid unchanged after pilot.'],
                      output_sha256={p.name:digest(p) for p in [planpath,worker,run/'all_fold_metrics.csv',run/'aggregate_metrics.csv',run/'per_seed_metrics.csv',run/'paired_differences.csv',run/'dbscan_fixed_radius_diagnostics.csv']})
        print('\nAna ortalamalar: temsil | yöntem | AP | F1',flush=True)
        for row in report['primary_results']:
            print(row['representation'],row['method_display_name'],round(row['AP_mean'],6),round(row['F1_mean'],6),flush=True)
    except KeyboardInterrupt:
        report.update(status='INTERRUPTED',active_job=active,completed_jobs=len(results),message='Aynı notebook Run All ile tamamlanan koşular korunarak devam eder.')
    except Exception as error:
        report.update(status='REVIEW_REQUIRED',active_job=active,completed_jobs=len(results),error=str(error))
    finally:
        report['elapsed_seconds_this_invocation']=time.monotonic()-started
        report['new_jobs_started_this_invocation']=executed
        write_json(summary_path,report);fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
    print('\nSON DURUM:',report['status'],'| tamamlanan:',report['completed_jobs'],'/30')
    print('Paylaşacağınız dosya:',summary_path.resolve())
    return report,summary_path


In [ ]:
PLAN_TEMPLATE = load_json('assets/05B/04_plan_template.json')

WORKER_SOURCE = load_text('assets/05B/04_worker_source.py')

result, summary_path = run_full(REVISION_ROOT, DATA_DIR, PLAN_TEMPLATE, WORKER_SOURCE, PYTHON_EXE)
